In [ ]:
# Install biobouncer from R-universe (safe to re-run).
if (!requireNamespace("biobouncer", quietly = TRUE)) {
  install.packages("biobouncer", repos = "https://samuelbharti.r-universe.dev")
}

# biobouncer (R): a gate for biological inputs

**biobouncer** validates biological identifiers (gene symbols, ontology terms,
variant formats, database accessions) through one small API, with the *same*
answers in R and Python.

Four validation modes:

| mode | what it checks | network |
|------|----------------|---------|
| `pattern`   | is the string well-formed? (regex/grammar) | offline |
| `cache`     | does the id exist in a pinned snapshot?    | offline |
| `remote`    | is the id live in the source API?          | online  |
| `existence` | snapshot first, then remote                | either  |

Everything below runs **offline** (pattern + cache) so it is safe to record.

In [ ]:
suppressMessages(library(biobouncer))
packageVersion("biobouncer")

## 1. Discover: what can biobouncer check?

In [ ]:
srcs <- sources()
cat(length(srcs), "sources supported\n\n")
srcs

In [ ]:
# Rich metadata for every source: id example, supported modes, species/version aware
source_info()

## 2. Snapshots: the bundled offline data

In [ ]:
cat("cache dir:", biobouncer_cache_dir(), "\n\n")
biobouncer_snapshots()

## 3. Mode `pattern`: is the string well-formed? (offline)

Format-only. Note the normalization + repair *suggestion* for a badly-cased id,
and that a well-formed GO id is correctly rejected when checked as MONDO.

In [ ]:
check_id(c("MONDO:0005148", "mondo:5148", "GO:0006915"), source_db = "mondo")

## 4. Mode `cache`: does the id actually exist? (offline snapshot)

Here we load a **real, messy** gene-disease association table and validate every
column. `cache` mode catches ids that are well-formed but wrong, and proposes the
current id for renamed/obsolete entries.

In [ ]:
df <- read.csv("data/associations.csv", stringsAsFactors = FALSE, na.strings = "")
df

In [ ]:
# One report per column: valid / repairable / invalid / missing.
cols <- list(gene = "hgnc", disease = "mondo", process = "go")
for (col in names(cols)) print(report_id(df[[col]], cols[[col]], how = "cache"))

In [ ]:
# Look inside one report: MLL -> KMT2A, CARS -> CARS1 are auto-suggested.
check_id(df$gene, source_db = "hgnc", how = "cache")

### Repair the whole table in one pass

In [ ]:
clean <- df
clean$gene    <- repair_id(df$gene,    "hgnc",  how = "cache")
clean$disease <- repair_id(df$disease, "mondo", how = "cache")
clean$process <- repair_id(df$process, "go",    how = "cache")
clean

## 5. `pattern` across many sources at once

A second real dataset mixes accessions from different databases. We validate each
row against its own `source_db`: UniProt, Ensembl, RefSeq, dbSNP, HGVS, ChEBI.

In [ ]:
ids <- read.csv("data/identifiers.csv", stringsAsFactors = FALSE)

res <- do.call(rbind, lapply(split(ids, ids$source_db), function(g)
  check_id(g$value, source_db = g$source_db[1], how = "pattern")))
res[, c("input", "source_db", "valid", "normalized", "suggestion")]

## 6. Mode `remote`: live check against the source API (optional)

Needs internet, so it is wrapped to fail gracefully during an offline recording.
`remote` is also species-aware for sources like Ensembl.

In [ ]:
tryCatch(
  check_id("ENSG00000139618", source_db = "ensembl",
           how = "remote", species = "homo_sapiens"),
  error = function(e) cat("remote skipped (offline):", conditionMessage(e), "\n"))

## 7. Drops into your validation stack

The same checks plug into common R validation frameworks, no re-implementation.

In [ ]:
# checkmate-style: check / assert / test
check_valid_id(c("MONDO:0005148", "mondo:5148"), "mondo")   # message on failure

# element-wise predicate for data-frame validation (assertr / validate)
is_mondo <- id_predicate("mondo")
is_mondo(c("MONDO:0005148", "MONDO:9999999"))

In [ ]:
# assert_valid_id() errors on bad input; here every id is valid, so it passes.
assert_valid_id(c("MONDO:0005148", "MONDO:0004975"), "mondo", how = "cache")
cat("assert passed: all disease ids valid\n")

One gate for biological inputs: offline by default, identical answers in Python.